<a href="https://colab.research.google.com/github/EstherMan05/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** _..._

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-01


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
vendors = pd.DataFrame(vendors_json)

joined = lines.merge(vendors, on='vendor_id', how='left',
                     indicator=True, validate='many_to_one')

print('rows:', len(joined), '(expected 4)')
print(joined['_merge'].value_counts())

unmatched = joined[joined['_merge'] == 'left_only']
if unmatched.empty:
    print('\nno unmatched vendors: every order vendor is in the vendor list')
else:
    print('\nunmatched vendor(s):', unmatched['vendor_id'].unique().tolist())
    print('units attached:', unmatched['qty'].sum())
joined

rows: 4 (expected 4)
_merge
both          4
left_only     0
right_only    0
Name: count, dtype: int64

no unmatched vendors: every order vendor is in the vendor list


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-01,Hoos Burgers,A,both


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
# Any unmatched vendor gets a visible label instead of silently dropping out
# of the groupby (groupby skips NaN keys by default). Hiding it would make the
# table total less than 7. Right now there are none, but this keeps the
# table honest if the data changes.
joined['name'] = joined['name'].fillna('Unknown vendor')

by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False)
print(by_vendor)
print('total:', by_vendor.sum())

name
Hoos Burgers    6
Cav Merch       1
Name: qty, dtype: int64
total: 7


### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# Same logic: label missing zones instead of letting them vanish
joined['zone'] = joined['zone'].fillna('Unassigned')

by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False)
print(by_zone)
print('zone total:', by_zone.sum(), '| line-item total:', lines['qty'].sum())

zone
A    7
Name: qty, dtype: int64
zone total: 7 | line-item total: 7


### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                                  # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},    # no qty
]

rows = []
for o in ragged_json:
    order_lines = o.get('lines')
    if not order_lines:                      # missing 'lines': keep the order, flag it
        rows.append({'order': o['order'], 'vendor_id': o['vendor_id'],
                     'item': None, 'qty': None})
    else:
        for ln in order_lines:
            rows.append({'order': o['order'], 'vendor_id': o['vendor_id'],
                         'item': ln.get('item'), 'qty': ln.get('qty')})

ragged = pd.DataFrame(rows)
ragged['qty_reported'] = ragged['qty'].notna()
ragged

,order,vendor_id,item,qty,qty_reported
0,4,V-01,Soda,2.0,True
1,5,V-10,None,NaN,False
2,6,V-01,Fries,NaN,False


### Q6 — Validate

In [7]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

A) One row per line item is the useful shape because the questions in this lab are about items and quantities, not orders. With one row per order, the lines would be a list stuck inside a cell, and totals per item, per vendor or per zone would need a loop. With one row per line item, they become a single groupby. The shape also makes the join safe to check for example 4 rows in, 4 rows out, and a quantity total that must stay at 7 before and after.

b) A missing quantity means "we don't know how many were sold." A quantity of zero means "we know none were sold." If you fill the missing ones with 0, the report states as fact something the data never said. Totals and averages get quietly dragged down. Fries would look like a zero-selling item instead of an unreported one, and the "no data" rows that should be chased up with the vendor become indistinguishable from real zeros. Keeping NaN and a qty_reported flag preserves the difference.